# A small grid

The bandit has one situation and four machines. There is no "what happens next", so gamma is 0. This grid has many situations. After a move, the update adds gamma times the best value of the next cell.

The grid is 4 by 4. You start at the top left. The goal, bottom right, pays +1. One pit, bottom row, pays -1 and ends the episode. Every other move costs 0.04. Moves that would leave the grid stay put. Seed 42.

This notebook is extra. The 15:10–16:00 block is still the bandit, and this notebook does not change its 20 pulls.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    marker = candidate / "lib" / "bandit.py"
    if marker.is_file() and (candidate / "pyproject.toml").is_file():
        root = candidate
        break
if root is None:
    raise FileNotFoundError("Start Jupyter from 03-reinforcement-learning.")
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("module", ROOT)


In [ ]:
import numpy as np

from gridworld import START, state_index, step, policy_arrows, reaches_goal

print("start", START, "goal bottom right, pit on the bottom row")


## The update

`q[state, action]` moves toward the target by a step of size alpha. The target is the reward, plus gamma times the best value of the next cell. When the episode is over, that next value is 0.

In [ ]:
def q_update(q, state, action, reward, next_state, done, alpha, gamma):
    target = None  # reward + gamma * best next value, or reward when done
    if target is None:
        raise NotImplementedError("Write the Q target.")
    q[state, action] += alpha * (target - q[state, action])


## The greedy policy

One action per cell: the column of Q with the largest value. Ties keep the lower index.

In [ ]:
def greedy(q):
    policy = None  # best action in each row of Q
    if policy is None:
        raise NotImplementedError("Read the best action from each row of Q.")
    return policy


In [ ]:
def train(gamma, episodes=400, seed=42):
    rng = np.random.default_rng(seed)
    q = np.zeros((16, 4))
    totals = []
    for _episode in range(episodes):
        state = state_index(*START)
        total = 0.0
        for _step in range(40):
            if rng.random() < 0.2:
                action = int(rng.integers(0, 4))
            else:
                action = int(np.argmax(q[state]))
            nxt, reward, done = step(state, action)
            q_update(q, state, action, reward, nxt, done, alpha=0.5, gamma=gamma)
            total += reward
            state = nxt
            if done:
                break
        totals.append(total)
    return q, np.asarray(totals)

q_plan, rewards_plan = train(0.9)
q_now, rewards_now = train(0.0)
print("gamma 0.9 first 50", round(float(rewards_plan[:50].mean()), 3))
print("gamma 0.9 last 50", round(float(rewards_plan[-50:].mean()), 3))
print(policy_arrows(greedy(q_plan)))
print("reaches goal", reaches_goal(greedy(q_plan)))
print("gamma 0 last 50", round(float(rewards_now[-50:].mean()), 3))
print(policy_arrows(greedy(q_now)))
print("reaches goal", reaches_goal(greedy(q_now)))


Write one sentence: which gamma reaches the goal, and does the later reward beat the early reward?